In [3]:
import pandas as pd

print("\n" + "="*50)
print("🔗 TIẾN HÀNH MERGE DỮ LIỆU MASTER VÀ LLM EXTRACTED")
print("="*50)

# 1. Khai báo đường dẫn (Đang đứng ở thư mục eda)
MASTER_FILE = '../data/html/coffee_master_dataset.csv'
LLM_FILE = '../data/html/llm_extracted.csv'
OUTPUT_FILE = '../data/html/final_enriched_dataset.csv'

try:
    # 2. Đọc dữ liệu
    print("📂 Đang tải dữ liệu từ kho...")
    df_master = pd.read_csv(MASTER_FILE)
    df_llm = pd.read_csv(LLM_FILE)

    print(f"📊 Số dòng Master Dataset: {len(df_master)}")
    print(f"📊 Số dòng LLM Extracted: {len(df_llm)}")

    # 3. Lọc file LLM (Chỉ lấy các hàng CÓ GIÁ TRỊ ở cột price_vnd)
    df_llm_filtered = df_llm.dropna(subset=['price_vnd']).copy()
    
    # Chắc chắn thêm 1 lớp lọc: Bỏ những dòng mà AI trả về chữ "NONE" hoặc rỗng
    df_llm_filtered = df_llm_filtered[df_llm_filtered['price_vnd'].astype(str).str.strip() != '']
    
    print(f"✅ Số dòng LLM sau khi lọc (Chỉ giữ lại dòng có Giá): {len(df_llm_filtered)}")

    # 4. Chuẩn hóa cột khóa
    df_master['url_hash'] = df_master['url_hash'].astype(str).str.strip().str.lower()
    df_llm_filtered['url_hash'] = df_llm_filtered['url_hash'].astype(str).str.strip().str.lower()

    # 5. Thực hiện Merge (Inner Join)
    print("🔗 Đang hợp nhất 2 bảng theo cột 'url_hash'...")
    df_merged = pd.merge(
        df_master, 
        df_llm_filtered, 
        on='url_hash', 
        how='inner' 
    )

    # 6. Sắp xếp và Đổi tên cột
    if 'date' in df_merged.columns:
        df_merged = df_merged.sort_values(by='date', ascending=False)
        
    df_merged = df_merged.rename(columns={'price_vnd': 'price_llm', 'exact_price': 'price_dl'})
    
    # =========================================================
    # 6.5. CHUẨN HÓA ĐỊNH DẠNG PRICE_LLM CHO GIỐNG PRICE_DL
    # =========================================================
    print("🪄 Đang đồng bộ định dạng cột price_llm...")
    def format_llm_price(val):
        try:
            # Ép kiểu về float, định dạng có dấu phẩy ngàn, sau đó thay phẩy thành chấm
            formatted = "{:,.0f}".format(float(val)).replace(',', '.')
            return f"{formatted} VNĐ/kg"
        except:
            return "N/A"

    df_merged['price_llm'] = df_merged['price_llm'].apply(format_llm_price)
    # =========================================================

    # 7. Xóa cột thừa và Lưu file
    df_merged = df_merged.drop(columns=['url_hash','is_coffee_price','direction','price_change','certainty','content_date','_status','_tokens_used'])
    
    df_merged.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

    print(f"\n🎉 HOÀN TẤT! Dữ liệu cuối cùng được gộp thành công với {len(df_merged)} bản ghi.")
    print(f"💾 Đã lưu file tại: {OUTPUT_FILE}")
    
    # In ra 3 dòng đầu để kiểm tra trực quan
    print("\n🔍 BẢN XEM TRƯỚC DỮ LIỆU ĐÃ GỘP (Top 3):")
    display(df_merged.head(3))

except FileNotFoundError as e:
    print(f"❌ Lỗi: Không tìm thấy file. Kiểm tra lại đường dẫn: {e}")
except Exception as e:
    print(f"❌ Lỗi hệ thống: {e}")


🔗 TIẾN HÀNH MERGE DỮ LIỆU MASTER VÀ LLM EXTRACTED
📂 Đang tải dữ liệu từ kho...
📊 Số dòng Master Dataset: 18191
📊 Số dòng LLM Extracted: 7048
✅ Số dòng LLM sau khi lọc (Chỉ giữ lại dòng có Giá): 4719
🔗 Đang hợp nhất 2 bảng theo cột 'url_hash'...
🪄 Đang đồng bộ định dạng cột price_llm...

🎉 HOÀN TẤT! Dữ liệu cuối cùng được gộp thành công với 15962 bản ghi.
💾 Đã lưu file tại: ../data/html/final_enriched_dataset.csv

🔍 BẢN XEM TRƯỚC DỮ LIỆU ĐÃ GỘP (Top 3):


,date,region,price_dl,target,domain,url,content_snippet,price_llm
0,2026-03-11,.,95.700 VNĐ/kg,robusta,vov.vn,https://vov.vn/thi-truong/gia-ca-phe-hom-nay-1...,Hiện giá cà phê đang dao động trong khoảng từ ...,95.700 VNĐ/kg
1,2026-03-11,Gia Lai,95.300 VNĐ/kg,robusta,vov.vn,https://vov.vn/thi-truong/gia-ca-phe-hom-nay-1...,"Chiều 11/3, giá cà phê tại tỉnh Gia Lai được g...",95.700 VNĐ/kg
2,2026-03-11,Gia Lai,95.700 VNĐ/kg,robusta,vov.vn,https://vov.vn/thi-truong/gia-ca-phe-hom-nay-1...,Giá cà phê tại tỉnh Gia Lai và Quảng Ngãi đang...,95.700 VNĐ/kg
